In [3]:
# Import pandas for working with tables and datasets
import pandas as pd

# Import numpy for numerical calculations
import numpy as np

# Import matplotlib for creating graphs
import matplotlib.pyplot as plt

# Import seaborn for easier statistical visualizations
import seaborn as sns

# Display more columns when showing DataFrames
pd.set_option("display.max_columns", None)

# Set a simple style for the graphs
sns.set_style("whitegrid")

# Problem 1 — Understanding the Home Credit Default Risk Competition

## 1. What kind of company is Home Credit?

Home Credit is a consumer-finance company that provides loans and other
financial services, including services for people who may have limited
or little traditional credit history.

The company uses information about applicants to help assess credit risk.

## 2. What prediction is required?

The competition asks participants to predict whether a loan applicant will
experience difficulty making the required payments.

The target variable in the application_train.csv dataset is:

- TARGET = 0: The application did not meet the competition's definition
  of payment difficulties.
- TARGET = 1: The application met the competition's definition of payment
  difficulties.

Therefore, this is a binary classification problem.

## 3. Possible benefits and risks of using these predictions

### Possible benefits

A prediction of credit risk could help a financial institution:

- identify applications that may require additional risk assessment
- improve consistency in credit-risk analysis
- estimate the probability of repayment difficulties
- potentially improve how financial resources are allocated

### Possible risks

There are also important risks:

- Predictions can contain errors.
- Historical data may contain biases or patterns that should not be treated
  as proof of causation.
- Sensitive information may create fairness and privacy concerns.
- An incorrect prediction could affect an individual's access to credit.
- Missing values and data-quality problems can affect the analysis.

Therefore, model predictions should be treated as statistical predictions,
not as proof that an individual person will or will not repay a loan.

### Source

Kaggle — Home Credit Default Risk Competition

https://www.kaggle.com/competitions/home-credit-default-risk

In [ ]:
# Upload the CSV files from the computer to the Colab session

from google.colab import files

uploaded = files.upload()

# Load the main Home Credit training dataset

df = pd.read_csv("application_train.csv")

# Display the first five rows
df.head()

# Check the number of rows and columns

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
#Problem 2
# Display the first five rows of the dataset

df.head()

# Display the last five rows

df.tail()

# Display information about the dataset

df.info()
# Display descriptive statistics for numerical columns

df.describe()

### Discussion

The describe() method provides statistical information such as:

- count
- mean
- standard deviation
- minimum
- 25th percentile
- median
- 75th percentile
- maximum

This helps us understand the distribution and scale of numerical variables.

Extreme minimum or maximum values may also indicate outliers that should
be investigated before further analysis.

In [ ]:
# Display summary statistics for categorical columns

df.describe(include="object")

# Count missing values in every column

missing_count = df.isnull().sum()

# Calculate the percentage of missing values in every column

missing_percentage = (df.isnull().mean() * 100)

# Create a table showing both values

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percentage": missing_percentage
})

# Sort from the highest missing percentage to the lowest

missing_summary = missing_summary.sort_values(
    by="Missing Percentage",
    ascending=False
)

# Display the top 30 columns with the most missing values

missing_summary.head(30)

### Discussion

Missing values are an important issue in this dataset.

The table shows both:

- the number of missing values
- the percentage of missing values

The percentage is useful because columns have the same number of rows,
making it easier to understand how much information is missing.

Columns with a very high percentage of missing values may require special
attention in later data preprocessing.

However, a missing value does not automatically mean that the variable
should be removed. Its meaning should first be understood from the
column description.

In [ ]:
# Display the values in the TARGET column

print(df["TARGET"].value_counts())

# Calculate the percentage of each target class

target_percentage = df["TARGET"].value_counts(normalize=True) * 100

print(target_percentage)

# Count the two target classes

target_counts = df["TARGET"].value_counts().sort_index()

# Create the graph

plt.figure(figsize=(7, 5))

ax = sns.barplot(
    x=target_counts.index,
    y=target_counts.values
)

plt.title("Distribution of TARGET")
plt.xlabel("TARGET")
plt.ylabel("Number of Applications")

# Add count and percentage labels above each bar

total = len(df)

for i, count in enumerate(target_counts.values):
    percentage = count / total * 100
    
    ax.text(
        i,
        count,
        f"{count:,}\n({percentage:.2f}%)",
        ha="center",
        va="bottom"
    )

plt.show()

### Discussion

The TARGET variable is the target of the competition.

TARGET = 0 represents applications that did not meet the competition's
definition of payment difficulties.

TARGET = 1 represents applications that met the competition's definition
of payment difficulties.

The graph shows both the number and percentage of applications in each
class.

The classes are highly imbalanced, meaning that the number of TARGET = 0
applications is much larger than TARGET = 1 applications.

# Problem 3 — Defining Analysis Questions

Based on the initial overview of the dataset, I will investigate the
following questions.

## Question 1 — Income and payment difficulties

How does AMT_INCOME_TOTAL differ between applications with TARGET = 0
and TARGET = 1?

### Why is this useful?

Income is one of the financial characteristics of an application.
Comparing its distribution between the target groups can help us understand
whether the groups have different income distributions.

This comparison does not prove that income causes payment difficulties.

---

## Question 2 — Credit amount relative to income

How does the requested credit amount (AMT_CREDIT) compare with the
applicant's total income (AMT_INCOME_TOTAL) for the two TARGET groups?

### Why is this useful?

A simple credit-to-income ratio can provide an additional way to examine
the relationship between the size of the requested credit and income.

This may help identify patterns that are useful for later analysis.

---

## Question 3 — Age and payment difficulties

How does applicant age, calculated from DAYS_BIRTH, differ between the
TARGET groups?

### Why is this useful?

Age is represented indirectly in the dataset as the number of days from
birth.

Examining its distribution can help us understand whether the two target
groups have different age distributions.

The result should be interpreted as an association, not as proof of causation.

---

## Question 4 — Income type and TARGET

How does TARGET vary across NAME_INCOME_TYPE categories?

### Why is this useful?

Different income types represent different employment or income situations.

Comparing the target distribution across these categories can reveal
patterns that may be useful for further investigation.

---

## Question 5 — Missing data

Which variables have the largest percentages of missing values?

### Why is this useful?

Missing values can affect data analysis and future machine-learning models.

Identifying variables with substantial missingness helps determine which
variables may require additional investigation or preprocessing.

---

## Additional question

During the exploration, I will also check whether the patterns observed
raise additional questions that should be investigated later.

In [ ]:
#PROBLEM 4 Data exploration
# Create a smaller copy containing the variables needed for this analysis

income_data = df[["AMT_INCOME_TOTAL", "TARGET"]].copy()

# Remove missing income values

income_data = income_data.dropna()

# Limit the graph to the 99th percentile to reduce the effect of extreme
# income values and make the main distribution easier to see

income_limit = income_data["AMT_INCOME_TOTAL"].quantile(0.99)

income_plot = income_data[
    income_data["AMT_INCOME_TOTAL"] <= income_limit
]

# Create the boxplot

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=income_plot,
    x="TARGET",
    y="AMT_INCOME_TOTAL"
)

plt.title("Income Distribution by TARGET")
plt.xlabel("TARGET")
plt.ylabel("Total Income")

plt.show()

### Exploration 1 — Income Distribution

This boxplot compares AMT_INCOME_TOTAL between TARGET = 0 and TARGET = 1.

The analysis excludes missing income values.

For visualization, values above the 99th percentile are excluded from the
graph. This does not delete them from the original dataset; it only makes
the main distribution easier to visualize.

The boxplot shows the median and spread of income within each target group.

### Limitation

The graph shows association only. It cannot establish that income causes
payment difficulties.

The removal of the highest 1% for visualization also means that the graph
does not show the complete income range.

In [ ]:
# Create a copy containing income and credit amount

credit_income = df[
    ["AMT_INCOME_TOTAL", "AMT_CREDIT", "TARGET"]
].copy()

# Remove rows where income or credit is missing

credit_income = credit_income.dropna()

# Calculate credit amount relative to income

credit_income["CREDIT_INCOME_RATIO"] = (
    credit_income["AMT_CREDIT"] /
    credit_income["AMT_INCOME_TOTAL"]
)

# Remove infinite values if any occur

credit_income = credit_income[
    np.isfinite(credit_income["CREDIT_INCOME_RATIO"])
]

# Limit the graph to the 99th percentile for readability

ratio_limit = credit_income["CREDIT_INCOME_RATIO"].quantile(0.99)

ratio_plot = credit_income[
    credit_income["CREDIT_INCOME_RATIO"] <= ratio_limit
]

# Create the graph

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=ratio_plot,
    x="TARGET",
    y="CREDIT_INCOME_RATIO"
)

plt.title("Credit-to-Income Ratio by TARGET")
plt.xlabel("TARGET")
plt.ylabel("Credit / Income Ratio")

plt.show()

### Exploration 2 — Credit-to-Income Ratio

The credit-to-income ratio was calculated as:

Credit-to-Income Ratio = AMT_CREDIT / AMT_INCOME_TOTAL

This provides a simple comparison between the requested credit amount and
the applicant's total income.

The graph compares the distribution of this ratio between TARGET = 0 and
TARGET = 1.

### Denominator

The denominator is AMT_INCOME_TOTAL.

### Limitation

This ratio is only a simple descriptive measure. It does not include
other financial obligations, expenses, household characteristics, loan
terms, or repayment history.

Therefore, it should not be interpreted as a complete measure of an
applicant's ability to repay.

In [ ]:
# Create a copy of the required columns

age_data = df[["DAYS_BIRTH", "TARGET"]].copy()

# Convert DAYS_BIRTH into approximate age in years

age_data["AGE_YEARS"] = (
    -age_data["DAYS_BIRTH"] / 365.25
)

# Remove missing values

age_data = age_data.dropna()

# Create the graph

plt.figure(figsize=(9, 5))

sns.histplot(
    data=age_data,
    x="AGE_YEARS",
    hue="TARGET",
    bins=30,
    kde=True,
    element="step"
)

plt.title("Approximate Age Distribution by TARGET")
plt.xlabel("Approximate Age (years)")
plt.ylabel("Number of Applications")

plt.show()

### Exploration 3 — Age Distribution

DAYS_BIRTH is represented as a negative number of days.

To make the variable easier to interpret, it was converted to approximate
age using:

Age = -DAYS_BIRTH / 365.25

The graph compares the age distributions of TARGET = 0 and TARGET = 1.

### Limitation

The calculated age is approximate because 365.25 days is used as an
average year.

The graph also shows distributions rather than proving that age causes
payment difficulties.

In [ ]:
# Create a table showing TARGET counts for each income type

income_type_table = pd.crosstab(
    df["NAME_INCOME_TYPE"],
    df["TARGET"]
)

income_type_table

# Calculate the percentage of TARGET = 1 within each income type

income_type_default_rate = (
    pd.crosstab(
        df["NAME_INCOME_TYPE"],
        df["TARGET"],
        normalize="index"
    ) * 100
)

# Select the TARGET = 1 percentage

income_type_default_rate = (
    income_type_default_rate[[1]]
    .rename(columns={1: "TARGET_1_Percentage"})
    .sort_values(
        "TARGET_1_Percentage",
        ascending=False
    )
)

income_type_default_rate

### Exploration 4 — Income Type

This analysis compares the percentage of TARGET = 1 applications within
each NAME_INCOME_TYPE category.

### Denominator

For each income type, the denominator is the number of applications
belonging to that income type.

This is important because simply comparing the number of TARGET = 1 cases
could be misleading when some categories contain many more applications
than others.

### Limitation

Some income-type categories may contain relatively few observations.

Therefore, percentages from small groups should be interpreted carefully.

The observed differences are associations in this dataset and should not
be interpreted as evidence that income type itself causes payment
difficulties.

In [ ]:
# Select the 20 columns with the highest percentage of missing values

top_missing = missing_summary.head(20).sort_values(
    "Missing Percentage",
    ascending=True
)

# Create a horizontal bar chart

plt.figure(figsize=(10, 8))

plt.barh(
    top_missing.index,
    top_missing["Missing Percentage"]
)

plt.title("Top 20 Columns by Missing-Value Percentage")
plt.xlabel("Missing Values (%)")
plt.ylabel("Column")

plt.show()

### Exploration 5 — Missing Values

This graph shows the 20 variables with the highest percentage of missing
values.

Missing values are an important data-quality issue because many statistical
methods and machine-learning algorithms cannot directly use missing values.

### Denominator

The percentage is calculated using the total number of rows in
application_train.csv.

### Limitation

A high percentage of missing values does not automatically mean that the
variable is useless.

The meaning of the variable should be checked using
HomeCredit_columns_description.csv before deciding how to handle it.

In [ ]:
# Select the 20 columns with the highest percentage of missing values

top_missing = missing_summary.head(20).sort_values(
    "Missing Percentage",
    ascending=True
)

# Create a horizontal bar chart

plt.figure(figsize=(10, 8))

plt.barh(
    top_missing.index,
    top_missing["Missing Percentage"]
)

plt.title("Top 20 Columns by Missing-Value Percentage")
plt.xlabel("Missing Values (%)")
plt.ylabel("Column")

plt.show()

### Exploration 5 — Missing Values

This graph shows the 20 variables with the highest percentage of missing
values.

Missing values are an important data-quality issue because many statistical
methods and machine-learning algorithms cannot directly use missing values.

### Denominator

The percentage is calculated using the total number of rows in
application_train.csv.

### Limitation

A high percentage of missing values does not automatically mean that the
variable is useless.

The meaning of the variable should be checked using
HomeCredit_columns_description.csv before deciding how to handle it.

In [ ]:
# Calculate the percentage of TARGET = 1 within each education category

education_target = (
    pd.crosstab(
        df["NAME_EDUCATION_TYPE"],
        df["TARGET"],
        normalize="index"
    ) * 100
)

# Keep only TARGET = 1

education_target = (
    education_target[[1]]
    .rename(columns={1: "TARGET_1_Percentage"})
    .sort_values(
        "TARGET_1_Percentage",
        ascending=True
    )
)

education_target

#Plot TARGET = 1 percentage by education type

plt.figure(figsize=(10, 6))

plt.barh(
    education_target.index,
    education_target["TARGET_1_Percentage"]
)

plt.title("TARGET = 1 Percentage by Education Type")
plt.xlabel("TARGET = 1 Percentage")
plt.ylabel("Education Type")

plt.show()

### Exploration 6 — Education Type

This graph compares the percentage of TARGET = 1 applications across
different education categories.

### Denominator

For each education category, the denominator is the number of applications
in that education category.

This makes the comparison a within-group percentage rather than a simple
count.

### Limitation

Education type is a categorical variable and the observed relationship
does not demonstrate causation.

The result may also be related to other variables in the dataset.
Therefore, this analysis should not be used by itself to make decisions
about individuals.

In [ ]:
# Select important numerical variables

summary_columns = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "DAYS_BIRTH",
    "TARGET"
]

# Display descriptive statistics

df[summary_columns].describe()

### Summary

The summary statistics provide an overall view of several important
numerical variables.

AMT_INCOME_TOTAL represents total income.

AMT_CREDIT represents the credit amount.

AMT_ANNUITY represents the annuity amount.

DAYS_BIRTH represents the applicant's birth date as a number of days
relative to the application date.

TARGET is the binary target variable.

In [ ]:
# Try to load the column description file

columns_description = pd.read_csv(
    "HomeCredit_columns_description.csv",
    encoding="latin1"
)

# Display the first rows

columns_description.head()

# Conclusion

In this assignment, I performed exploratory data analysis on the Home
Credit Default Risk application_train.csv dataset.

First, I examined the purpose of the competition and identified TARGET as
the binary target variable representing payment difficulties according to
the competition definition.

Next, I examined the structure of the dataset using head(), info(), and
describe(). I also investigated categorical variables and calculated
missing-value counts and percentages.

The target distribution showed that the two classes are imbalanced, with
TARGET = 0 representing a much larger portion of the training data than
TARGET = 1.

I then defined several analysis questions involving income, credit amount,
age, income type, education type, and missing values.

The exploratory analysis showed that the dataset contains differences in
the distributions of several variables between target groups. I also
identified substantial missing data in some variables.

However, these observations represent associations in the dataset and
should not be interpreted as proof of causation. Further analysis would be
needed to understand relationships between variables and to determine how
the data should be prepared for machine-learning models.

The EDA provides a foundation for the next stage of the project, where
machine-learning models can be trained and evaluated.